# 05 — EXP_06: Question complexity labels (rule-based, no LLM)

Applies `src/retrieval/complexity.py` to all 12,723 MedQA-US questions and writes `data/processed/complexity_labels.parquet`.

**The rule** (anchored to full-corpus 33rd / 67th percentiles, computed 2026-05-10):
- `Complex`  — long stem (≥133 words) AND many MetaMap entities (≥41), **or** any clinical-decision cue word ("best next step", "initial management", …)
- `Simple`   — short stem (≤93 words) AND few entities (≤28), **or** a factoid/mechanism cue word with a short stem
- `Moderate` — everything else

**Why a rule, not a learned classifier**: see `src/retrieval/complexity.py` module docstring. Short version: transparent, viva-defensible, no model-of-a-model dependency.

**Caveat for the methodology footnote**: MedQA is overwhelmingly clinical vignettes — even the `Simple` bucket is mostly *short-vignette* questions, not pure factoids. The proposal's terminology (Simple / Moderate / Complex) is preserved for plan-alignment but the rule is honestly a *length + entity density + cue-word* proxy.

**Routing (set in EXP_07)**:

| Bucket | Variant A (proposal) | Variant B (data-driven) |
|---|---|---|
| Simple   | Naive       | No-RAG    |
| Moderate | Hybrid      | Multi-Hop |
| Complex  | Multi-Hop   | Multi-Hop |

Both variants will be run in EXP_07 and compared in Table 10.

## 1. Setup

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import json

import pandas as pd

from src.data.loaders import load_medqa_4opt
from src.retrieval.complexity import (
    LABELS,
    PHRASES_P33,
    PHRASES_P67,
    WORDS_P33,
    WORDS_P67,
    classify_dataframe,
)

print("Repo root:", REPO_ROOT)
print(f"Thresholds: words {WORDS_P33}/{WORDS_P67}, phrases {PHRASES_P33}/{PHRASES_P67}")

Repo root: /Users/rajak/Workstation/Projects/myGitHub/thesis-project
Thresholds: words 93/133, phrases 28/41


## 2. Load data + apply classifier

In [2]:
md = load_medqa_4opt()
md = md.reset_index(drop=False).rename(columns={"index": "row_idx"})
md["question_id"] = "medqa_" + md["row_idx"].astype(str)
print(f"Loaded {len(md):,} questions")

labelled = classify_dataframe(md)
print("\n=== Distribution (n = 12,723) ===")
vc = labelled["complexity"].value_counts()
for lbl in LABELS:
    n = vc.get(lbl, 0)
    print(f"  {lbl:8s}: {n:>5,}  ({100 * n / len(labelled):5.1f} %)")

Loaded 12,723 questions

=== Distribution (n = 12,723) ===
  Simple  : 3,759  ( 29.5 %)
  Moderate: 4,163  ( 32.7 %)
  Complex : 4,801  ( 37.7 %)


**Acceptance gate (distribution)** — all three buckets should land in the **15–55 %** range. If one bucket is < 10 % the rule is too narrow; if > 60 % it's too broad. Re-tune thresholds in `src/retrieval/complexity.py` (`WORDS_P33`/`WORDS_P67`/`PHRASES_P33`/`PHRASES_P67`) and re-run.

## 3. Stratified breakdown — by USMLE step + by split

In [4]:
print("=== By USMLE step ===")
step_xtab = (
    labelled.groupby(["meta_info", "complexity"]).size().unstack(fill_value=0)[list(LABELS)]
)
print(step_xtab)
print("  (proportions per step)")
print((step_xtab.div(step_xtab.sum(axis=1), axis=0) * 100).round(1))

print("\n=== By split ===")
split_xtab = (
    labelled.groupby(["split", "complexity"]).size().unstack(fill_value=0)[list(LABELS)]
)
print(split_xtab)

=== By USMLE step ===
complexity  Simple  Moderate  Complex
meta_info                            
step1         3252      2557     1200
step2&3        507      1606     3601
  (proportions per step)
complexity  Simple  Moderate  Complex
meta_info                            
step1         46.4      36.5     17.1
step2&3        8.9      28.1     63.0

=== By split ===
complexity  Simple  Moderate  Complex
split                                
dev            362       426      484
test           366       394      513
train         3031      3343     3804


**Expected pattern**: step2&3 (clinical decision) should skew Complex; step1 (basic science) should skew Simple/Moderate. Test split distribution should match overall (rule is content-only, not split-aware).

## 4. Per-bucket accuracy on test_1273 — informative for EXP_07 routing

This cell is **read-only EDA** — it doesn't affect EXP_06 outputs. It loads the EXP_01–EXP_05 prediction files and reports per-bucket accuracy so the routing decision in EXP_07 has data to anchor on.

In [5]:
test = labelled[labelled.split == "test"][["question_id", "complexity"]]

archs = [
    ("NoRAG", "exp_01_base_llm"),
    ("Naive", "exp_02_naive_rag"),
    ("Sparse", "exp_03_sparse_rag"),
    ("Hybrid", "exp_04_hybrid_rag"),
    ("MultiHop", "exp_05_multi_hop_rag"),
]

merged = test
for label, prefix in archs:
    fp = REPO_ROOT / f"results/{prefix}__test_1273/predictions.jsonl"
    if not fp.exists():
        print(f"  skipping {label} — {fp} not found")
        continue
    p = pd.DataFrame([json.loads(line) for line in fp.read_text().splitlines()])
    merged = merged.merge(
        p[["question_id", "is_correct"]].rename(columns={"is_correct": label}),
        on="question_id",
    )

agg = merged.groupby("complexity").agg(
    n=("question_id", "count"),
    NoRAG=("NoRAG", "mean"),
    Naive=("Naive", "mean"),
    Sparse=("Sparse", "mean"),
    Hybrid=("Hybrid", "mean"),
    MultiHop=("MultiHop", "mean"),
).round(4)
print("=== Acuuracy by complexity bucket on test_1273 ===")
print(agg.reindex(list(LABELS)))

# Routing-table simulator (per-question pick from the table; not the same as
# running EXP_07, which actually invokes the chosen retriever per question —
# but a useful upper-bound check that the table is coherent).
for variant, table in [
    ("A (proposal)", {"Simple": "Naive", "Moderate": "Hybrid", "Complex": "MultiHop"}),
    ("B (data-driven)", {"Simple": "NoRAG", "Moderate": "MultiHop", "Complex": "MultiHop"}),
]:
    sim = merged.apply(lambda r: r[table[r["complexity"]]], axis=1)
    acc = sim.mean()
    print(f"\n  Variant {variant}  simulated acc = {acc:.4f}  ({sim.sum()}/{len(sim)})")

print(f"  ----- comparison -----")
print(f"  All No-RAG (EXP_01)   = {merged.NoRAG.mean():.4f}")
print(f"  All Multi-Hop (EXP_05)= {merged.MultiHop.mean():.4f}")
print(f"  Oracle (best per Q)   = {merged[['NoRAG','Naive','Sparse','Hybrid','MultiHop']].max(axis=1).mean():.4f}")

=== Acuuracy by complexity bucket on test_1273 ===
              n   NoRAG   Naive  Sparse  Hybrid  MultiHop
complexity                                               
Simple      366  0.7951  0.8169  0.7760  0.8060    0.8361
Moderate    394  0.7563  0.7437  0.7589  0.7690    0.7716
Complex     513  0.7719  0.7251  0.7446  0.7349    0.7856

  Variant A (proposal)  simulated acc = 0.7895  (1005/1273)

  Variant B (data-driven)  simulated acc = 0.7840  (998/1273)
  ----- comparison -----
  All No-RAG (EXP_01)   = 0.7738
  All Multi-Hop (EXP_05)= 0.7958
  Oracle (best per Q)   = 0.8712


**Interpretation guide** for EXP_07 routing-table choice:
- The **simulated** numbers above are reusing existing predictions (cheap; just a join). Real EXP_07 runs each question through its assigned retriever and may differ slightly because of cache misses on cross-architecture chunks. The simulation is a tight upper bound on "what each routing table can achieve".
- If Variant A simulates to ≥ all-Multi-Hop (0.7958), the proposal table is data-defensible.
- If Variant B simulates higher than Variant A, the data-driven binary table wins.
- Either way, EXP_07 will run both and report both rows in Table 10.

## 5. Manual review — sample 100 random rows for human eyeball

Per [`plan.md §7`](../plan.md), the acceptance criterion for EXP_06 is **≥ 80 % rater agreement** on a 100-row manual review. The next cell prints 100 randomly sampled questions with their assigned bucket so the reviewer can mark each one.

**How to review**: scan the printout, mentally classify each as Simple / Moderate / Complex, and tick those where you disagree with the rule. If disagreement count > 20 / 100, re-tune the thresholds in `src/retrieval/complexity.py`.

In [6]:
REVIEW_SAMPLE_SIZE = 100
REVIEW_SEED = 42

review = labelled.sample(n=REVIEW_SAMPLE_SIZE, random_state=REVIEW_SEED)[
    ["question_id", "complexity", "n_words", "n_phrases", "has_complex_cue", "has_simple_cue", "meta_info", "split", "question"]
].reset_index(drop=True)

# Stratified sample: 33 / 33 / 34 from each bucket (force balance for a fairer review)
by_bucket = []
for lbl, n in zip(LABELS, [33, 33, 34]):
    pool = labelled[labelled.complexity == lbl]
    if len(pool) >= n:
        by_bucket.append(pool.sample(n=n, random_state=REVIEW_SEED))
review = pd.concat(by_bucket).sample(frac=1, random_state=REVIEW_SEED).reset_index(drop=True)

print(f"Manual review sample: {len(review)} rows (33+33+34 stratified, seed={REVIEW_SEED})\n")
for i, row in review.iterrows():
    q = row.question[:200].replace("\n", " ")
    if len(row.question) > 200:
        q += "…"
    cues = []
    if row.has_complex_cue:
        cues.append("COMPLEX_CUE")
    if row.has_simple_cue:
        cues.append("SIMPLE_CUE")
    cue_str = f" [{','.join(cues)}]" if cues else ""
    print(f"#{i+1:3d} {row.question_id:>12s} | {row.complexity:>8s} | w={row.n_words:>3d} p={row.n_phrases:>3d}{cue_str} | {row.meta_info}/{row.split}")
    print(f"     {q}\n")

Manual review sample: 100 rows (33+33+34 stratified, seed=42)

#  1   medqa_2670 |  Complex | w=145 p= 44 | step1/train
     A 9-year-old boy is brought to the emergency room by his mother for weakness, diaphoresis, and syncope. His mother says that he has never been diagnosed with any medical conditions but has been having…

#  2   medqa_4808 | Moderate | w=102 p= 27 | step2&3/train
     A 54-year-old man comes to the emergency department because of episodic palpitations for the past 12 hours. He has no chest pain. He has coronary artery disease and type 2 diabetes mellitus. His curre…

#  3     medqa_25 |  Complex | w=155 p= 48 | step2&3/train
     A 53-year-old man comes to the emergency department because of severe right-sided flank pain for 3 hours. The pain is colicky, radiates towards his right groin, and he describes it as 8/10 in intensit…

#  4   medqa_1173 | Moderate | w=119 p= 35 | step1/train
     A 24-year-old woman presents to her gynecologist complaining of mild pelvic 

**Acceptance gate (manual review)** — record the rater-disagreement count below before proceeding to §6.

```
rater_disagreement = ?? / 100
```

If ≤ 20: ✓ proceed to §6 (write parquet).

If > 20: re-tune `src/retrieval/complexity.py` thresholds and re-run from §2.

## 6. Write `data/processed/complexity_labels.parquet`

In [7]:
OUT_PATH = REPO_ROOT / "data/processed/complexity_labels.parquet"

out = labelled[
    [
        "question_id",
        "complexity",
        "n_words",
        "n_phrases",
        "has_complex_cue",
        "has_simple_cue",
        "meta_info",
        "split",
    ]
].copy()

# Make complexity an ordered categorical so downstream groupbys order Simple→Complex.
out["complexity"] = pd.Categorical(out["complexity"], categories=list(LABELS), ordered=True)

out.to_parquet(OUT_PATH, index=False)
print(f"✓ Wrote {OUT_PATH}")
print(f"  rows: {len(out):,}")
print(f"  size: {OUT_PATH.stat().st_size / 1024:.1f} KB")
print(f"  cols: {list(out.columns)}")

✓ Wrote /Users/rajak/Workstation/Projects/myGitHub/thesis-project/data/processed/complexity_labels.parquet
  rows: 12,723
  size: 123.0 KB
  cols: ['question_id', 'complexity', 'n_words', 'n_phrases', 'has_complex_cue', 'has_simple_cue', 'meta_info', 'split']


---

**Next.** EXP_07 — adaptive RAG controller. Build `src/retrieval/adaptive.py` (dispatcher) + `notebooks/05_exp07_adaptive_rag.ipynb` (runs both routing-table variants on test_1273, score-joins RAGAS from underlying architectures' golden_234, populates Table 10).